# What four weeks of disciplined negative results look like

### Playground Series S6E8, smartphone addiction

Best cross-validation **0.963880 +/- 0.000555** (5-seed blend); best public LB
**0.965090** (3-seed blend). Those are two different submissions, and the fact
that they are is one of the things this notebook is about.

This is not a solution notebook. There is no 55-model stack here and nothing in
it will move you up the leaderboard by itself. It is a record of which ideas
worked, which did not, and — the part I think is actually worth reading — how
each one was killed cheaply enough that the next one still had time to be tried.

**The short version:** the entire competition, from an untuned baseline to my best
model, was worth **+0.0089 AUC**. Every point of it came from model capacity and
stochastic averaging. Not one engineered feature ever helped. Neither did a second
gradient-boosting library. My highest-priority idea died in ninety seconds and
cost nothing, which is the only reason there was time for the ones that worked.

Four things in here that I have not seen written up elsewhere for this dataset:

1. **Rank correlation between two models does not predict whether blending them
   helps.** Measured across all 66 pairs of models I trained. This is the opposite
   of the usual advice, and it nearly cost me a day of CatBoost runs.
2. **Fold standard deviation is the wrong bar for a paired comparison** and it
   almost buried my one real improvement.
3. **The public leaderboard here has a standard error near 0.001**, which is larger
   than every gain after the fourth experiment. Two of my submissions demonstrate
   this cleanly.
4. **The missingness in this data is injected at random with respect to the
   target**, with the twelve-feature test to show it.

The full experiment ledger, every notebook, and the working log are in the repo
linked at the bottom.

## Setup

The measured results below are inlined as data. They come from the out-of-fold
prediction vectors saved by the training runs in the repo, not from numbers typed
in by hand — the script that produced this blob recomputes every figure from those
vectors, so a number in a chart and a number in the ledger cannot drift apart.

One seed of the final model is retrained live further down, so you can watch a
ledger row reproduce instead of taking the table on trust.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Measured from the saved out-of-fold vectors. See the repo link at the bottom;
# `writeup_numbers.py` regenerates this blob from `artifacts/oof/`.
D = json.loads(r'''{"pairs":[{"a":"anchor (100 trees)","b":"300 trees","spearman":0.9930765017385362,"cv_gap":0.005658141349606405,"gain":-0.0021846019231697156,"folds_won":0},{"a":"anchor (100 trees)","b":"1000 trees","spearman":0.9836829150135086,"cv_gap":0.007194204106504065,"gain":-0.002095635420481301,"folds_won":0},{"a":"anchor (100 trees)","b":"2000 trees","spearman":0.9745648778947578,"cv_gap":0.006885622876645847,"gain":-0.0015462840204955696,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.10","spearman":0.984439257835441,"cv_gap":0.007251535687893829,"gain":-0.0020525769354049483,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.05","spearman":0.9890902353771174,"cv_gap":0.00826312619309244,"gain":-0.002770141125892489,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.03","spearman":0.9888691181011995,"cv_gap":0.008327838102298424,"gain":-0.0028360939844481294,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 42","spearman":0.9883336427547597,"cv_gap":0.008523831677051286,"gain":-0.0028259452733267352,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 2024","spearman":0.9834789992708939,"cv_gap":0.008286941005899218,"gain":-0.0026049963785901966,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 7","spearman":0.9870274795339734,"cv_gap":0.008498727025603947,"gain":-0.0027421523495735345,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 2025","spearman":0.9856464416771401,"cv_gap":0.00839074743423518,"gain":-0.0026766225107882403,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 13","spearman":0.9876338741521341,"cv_gap":0.008536002331981485,"gain":-0.0028060978845790173,"folds_won":0},{"a":"300 trees","b":"1000 trees","spearman":0.9885054227814063,"cv_gap":0.0015360627568976604,"gain":-0.00019118865122798035,"folds_won":1},{"a":"300 trees","b":"2000 trees","spearman":0.9787488030888335,"cv_gap":0.0012274815270394424,"gain":0.0002832391139844681,"folds_won":5},{"a":"300 trees","b":"lr 0.10","spearman":0.9897116101269284,"cv_gap":0.0015933943382874238,"gain":-0.00016800966362118253,"folds_won":1},{"a":"300 trees","b":"lr 0.05","spearman":0.9928298718287766,"cv_gap":0.0026049848434860357,"gain":-0.0008315293646048883,"folds_won":0},{"a":"300 trees","b":"lr 0.03","spearman":0.9925160458419798,"cv_gap":0.002669696752692019,"gain":-0.0008907828275825702,"folds_won":0},{"a":"300 trees","b":"bagged seed 42","spearman":0.9917988890272705,"cv_gap":0.002865690327444881,"gain":-0.0008975168556043967,"folds_won":0},{"a":"300 trees","b":"bagged seed 2024","spearman":0.9875189229544078,"cv_gap":0.002628799656292813,"gain":-0.0006527027918643569,"folds_won":1},{"a":"300 trees","b":"bagged seed 7","spearman":0.9910458358371231,"cv_gap":0.0028405856759975423,"gain":-0.0007902378670594957,"folds_won":0},{"a":"300 trees","b":"bagged seed 2025","spearman":0.9896924463922334,"cv_gap":0.0027326060846287747,"gain":-0.0007258545206625833,"folds_won":0},{"a":"300 trees","b":"bagged seed 13","spearman":0.9915065958175602,"cv_gap":0.00287786098237508,"gain":-0.000848011230202217,"folds_won":0},{"a":"1000 trees","b":"2000 trees","spearman":0.9838366777610164,"cv_gap":0.000308581229858218,"gain":0.0005035079836765322,"folds_won":5},{"a":"1000 trees","b":"lr 0.10","spearman":0.993022042353452,"cv_gap":5.7331581389763464e-05,"gain":0.00038876154675240306,"folds_won":4},{"a":"1000 trees","b":"lr 0.05","spearman":0.988195036539881,"cv_gap":0.0010689220865883753,"gain":-6.606134162769894e-05,"folds_won":1},{"a":"1000 trees","b":"lr 0.03","spearman":0.9872628803978186,"cv_gap":0.0011336339957943586,"gain":-0.00012190562459422072,"folds_won":1},{"a":"1000 trees","b":"bagged seed 42","spearman":0.986384154208615,"cv_gap":0.0013296275705472205,"gain":-0.00011715093433148916,"folds_won":1},{"a":"1000 trees","b":"bagged seed 2024","spearman":0.9857334506794263,"cv_gap":0.0010927368993951525,"gain":0.00015178167196217007,"folds_won":2},{"a":"1000 trees","b":"bagged seed 7","spearman":0.9855514711213312,"cv_gap":0.001304522919099882,"gain":3.0640205427756586e-06,"folds_won":2},{"a":"1000 trees","b":"bagged seed 2025","spearman":0.9862711547093388,"cv_gap":0.0011965433277311144,"gain":6.81779260263582e-05,"folds_won":2},{"a":"1000 trees","b":"bagged seed 13","spearman":0.98603856433902,"cv_gap":0.0013417982254774197,"gain":-5.317186754929537e-05,"folds_won":1},{"a":"2000 trees","b":"lr 0.10","spearman":0.9833150022204985,"cv_gap":0.0003659128112479815,"gain":0.0004936151298087887,"folds_won":5},{"a":"2000 trees","b":"lr 0.05","spearman":0.979252069284783,"cv_gap":0.0013775033164465933,"gain":-5.071948560479989e-05,"folds_won":2},{"a":"2000 trees","b":"lr 0.03","spearman":0.9784559497000986,"cv_gap":0.0014422152256525766,"gain":-0.00010150940902369232,"folds_won":2},{"a":"2000 trees","b":"bagged seed 42","spearman":0.9778607195685959,"cv_gap":0.0016382088004054385,"gain":-0.00010393573187614802,"folds_won":2},{"a":"2000 trees","b":"bagged seed 2024","spearman":0.9774052957938391,"cv_gap":0.0014013181292533705,"gain":0.00017099831572446876,"folds_won":3},{"a":"2000 trees","b":"bagged seed 7","spearman":0.9772698905921975,"cv_gap":0.0016131041489581,"gain":2.3165383544276884e-05,"folds_won":2},{"a":"2000 trees","b":"bagged seed 2025","spearman":0.9773175578834753,"cv_gap":0.0015051245575893324,"gain":8.744786730061627e-05,"folds_won":3},{"a":"2000 trees","b":"bagged seed 13","spearman":0.9776912226772052,"cv_gap":0.0016503794553356377,"gain":-3.298094121246819e-05,"folds_won":2},{"a":"lr 0.10","b":"lr 0.05","spearman":0.9892230127072061,"cv_gap":0.0010115905051986118,"gain":-1.789765781663455e-05,"folds_won":1},{"a":"lr 0.10","b":"lr 0.03","spearman":0.9880457011882899,"cv_gap":0.0010763024144045952,"gain":-7.345016338087263e-05,"folds_won":1},{"a":"lr 0.10","b":"bagged seed 42","spearman":0.9875360821450077,"cv_gap":0.001272295989157457,"gain":-6.644238074391407e-05,"folds_won":1},{"a":"lr 0.10","b":"bagged seed 2024","spearman":0.9869777426626942,"cv_gap":0.001035405318005389,"gain":0.00020431856153169115,"folds_won":3},{"a":"lr 0.10","b":"bagged seed 7","spearman":0.9865373605534884,"cv_gap":0.0012471913377101185,"gain":5.5054374122454064e-05,"folds_won":3},{"a":"lr 0.10","b":"bagged seed 2025","spearman":0.9873094081837185,"cv_gap":0.001139211746341351,"gain":0.00012052103314810214,"folds_won":2},{"a":"lr 0.10","b":"bagged seed 13","spearman":0.9872940895721637,"cv_gap":0.0012844666440876562,"gain":-2.4306118975081502e-06,"folds_won":2},{"a":"lr 0.05","b":"lr 0.03","spearman":0.998091892385352,"cv_gap":6.471190920598335e-05,"gain":7.230077590392181e-05,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 42","spearman":0.9957503870292943,"cv_gap":0.0002607054839588452,"gain":0.00010234125452337483,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 2024","spearman":0.9913523243456657,"cv_gap":2.381481280677722e-05,"gain":0.00036753162949680894,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 7","spearman":0.994866027966549,"cv_gap":0.00023560083251150665,"gain":0.00022474088739101727,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 2025","spearman":0.9931959904996398,"cv_gap":0.0001276212411427391,"gain":0.00028928505220238154,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 13","spearman":0.9947631273975702,"cv_gap":0.0002728761388890444,"gain":0.00017169301372617075,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 42","spearman":0.9961549651143419,"cv_gap":0.00019599357475286183,"gain":0.00011020117395905693,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 2024","spearman":0.990550712129159,"cv_gap":4.0897096399206134e-05,"gain":0.00033676062731287094,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 7","spearman":0.9947482610126726,"cv_gap":0.0001708889233055233,"gain":0.00023327687153544828,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 2025","spearman":0.992860946772584,"cv_gap":6.290933193675574e-05,"gain":0.0002982862914564066,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 13","spearman":0.9948778894230152,"cv_gap":0.00020816422968306103,"gain":0.00017952668078475843,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 2024","spearman":0.989781705992632,"cv_gap":0.00023689067115206797,"gain":0.00023829578475076385,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 7","spearman":0.9952482999788735,"cv_gap":2.5104651447338533e-05,"gain":0.0002993946696464134,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 2025","spearman":0.9925200396846862,"cv_gap":0.0001330842428161061,"gain":0.0002637611176472099,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 13","spearman":0.9963487474204312,"cv_gap":1.2170654930199198e-05,"gain":0.00027609706911186916,"folds_won":5},{"a":"bagged seed 2024","b":"bagged seed 7","spearman":0.9902793175540823,"cv_gap":0.00021178601970472943,"gain":0.0002741622578831704,"folds_won":4},{"a":"bagged seed 2024","b":"bagged seed 2025","spearman":0.992782079445031,"cv_gap":0.00010380642833596188,"gain":0.00036036872088816006,"folds_won":5},{"a":"bagged seed 2024","b":"bagged seed 13","spearman":0.9899107954603206,"cv_gap":0.00024906132608226716,"gain":0.0002250819357786149,"folds_won":4},{"a":"bagged seed 7","b":"bagged seed 2025","spearman":0.9920436551301872,"cv_gap":0.00010797959136876756,"gain":0.0003094607507363234,"folds_won":5},{"a":"bagged seed 7","b":"bagged seed 13","spearman":0.9955129368789487,"cv_gap":3.727530637753773e-05,"gain":0.0002629042287467298,"folds_won":5},{"a":"bagged seed 2025","b":"bagged seed 13","spearman":0.9926527702006471,"cv_gap":0.0001452548977463053,"gain":0.00025114728014290487,"folds_won":5}],"missingness":[{"feature":"gender","miss_frac":0.041995,"lift":-0.000662,"z":-0.243335},{"feature":"stress_level","miss_frac":0.079766,"lift":-0.00046,"z":-0.228135},{"feature":"social_media_hours","miss_frac":0.193811,"lift":-1.2e-05,"z":-0.008802},{"feature":"academic_work_impact","miss_frac":0.063966,"lift":3.4e-05,"z":0.015331},{"feature":"notifications_per_day","miss_frac":0.097754,"lift":0.000906,"z":0.492943},{"feature":"work_study_hours","miss_frac":0.074516,"lift":0.001319,"z":0.634227},{"feature":"gaming_hours","miss_frac":0.183435,"lift":0.001324,"z":0.938332},{"feature":"weekend_screen_time","miss_frac":0.162089,"lift":0.001467,"z":0.990331},{"feature":"daily_screen_time_hours","miss_frac":0.138644,"lift":0.002178,"z":1.378473},{"feature":"app_opens_per_day","miss_frac":0.116739,"lift":0.003807,"z":2.238488},{"feature":"age","miss_frac":0.041843,"lift":0.004151,"z":1.522202},{"feature":"sleep_hours","miss_frac":0.064336,"lift":0.004224,"z":1.898053}],"seed_saturation":[{"n_seeds":1,"cv":0.9634705327026813,"sd":0.0005907357588788866},{"n_seeds":2,"cv":0.9637088284874322,"sd":0.0006021893977001631},{"n_seeds":3,"cv":0.9638210471372626,"sd":0.0005601863998334479},{"n_seeds":4,"cv":0.9638575079829785,"sd":0.0005564493204720445},{"n_seeds":5,"cv":0.9638804130611953,"sd":0.0005547070146738952}],"ledger":[{"id":1,"name":"lgbm_default_anchor","cv_mean":0.954947,"cv_std":0.000645,"lb_public":0.95594},{"id":2,"name":"lgbm_trees100","cv_mean":0.954947,"cv_std":0.000645,"lb_public":null},{"id":3,"name":"lgbm_trees300","cv_mean":0.960605,"cv_std":0.000688,"lb_public":null},{"id":4,"name":"lgbm_trees1000","cv_mean":0.962141,"cv_std":0.000859,"lb_public":0.96435},{"id":5,"name":"lgbm_trees2000","cv_mean":0.961832,"cv_std":0.000952,"lb_public":null},{"id":6,"name":"lgbm_lr01","cv_mean":0.962198,"cv_std":0.000816,"lb_public":null},{"id":7,"name":"lgbm_lr005","cv_mean":0.96321,"cv_std":0.000591,"lb_public":null},{"id":8,"name":"lgbm_lr003","cv_mean":0.963275,"cv_std":0.000549,"lb_public":0.96478},{"id":9,"name":"lgbm_bag08_seed42","cv_mean":0.963471,"cv_std":0.000591,"lb_public":null},{"id":10,"name":"lgbm_bag08_seed2024","cv_mean":0.963234,"cv_std":0.000899,"lb_public":null},{"id":11,"name":"lgbm_bag08_seed7","cv_mean":0.963445,"cv_std":0.000478,"lb_public":null},{"id":12,"name":"lgbm_bag08_seedblend3","cv_mean":0.963821,"cv_std":0.00056,"lb_public":0.96509},{"id":13,"name":"lgbm_bag08_seed2025","cv_mean":0.963337,"cv_std":0.000731,"lb_public":null},{"id":14,"name":"lgbm_bag08_seed13","cv_mean":0.963483,"cv_std":0.000552,"lb_public":null},{"id":15,"name":"lgbm_bag08_seedblend5","cv_mean":0.96388,"cv_std":0.000555,"lb_public":0.96508}]}''')

print(f"{len(D['pairs'])} model pairs, {len(D['ledger'])} ledger rows, "
      f"{len(D['missingness'])} features")

### Chart styling

Two colours, checked for colour-vision deficiency separation against a white
background rather than picked by eye. Everything else is recessive on purpose:
hairline grid, no top or right spine, labels in grey so the marks carry the ink.

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, SECOND, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#ffffff"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": SECOND, "axes.edgecolor": AXIS,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "figure.dpi": 120,
})


def finish(ax, title, sub=None):
    """Title and subtitle stacked above the axes.

    Placed by hand rather than with set_title, which collides with a subtitle
    drawn just above the axes.
    """
    n = sub.count("\n") + 1 if sub else 0
    ax.text(0, 1.06 + 0.055 * n, title, transform=ax.transAxes, color=INK,
            fontsize=12, fontweight="bold", va="bottom")
    if sub:
        ax.text(0, 1.03, sub, transform=ax.transAxes, color=SECOND,
                fontsize=9.5, va="bottom", linespacing=1.4)
    ax.set_axisbelow(True)


print("styled")

## First: the metric and the validation scheme

Before looking at a single feature. The metric is ROC AUC on a predicted
probability, which decides three things at once: the loss is log loss, the
predictions never need calibrating because AUC only reads their order, and rank
averaging is the natural way to combine models.

The split is **5-fold stratified, shuffled, seed 42**, and it never changed. There
are no groups, no repeated entities and no time index in this data, so stratifying
on the target is the whole requirement. Every number in this notebook comes from
that one split, which is what makes them comparable to each other.

The thing worth copying is not the split. It is that **the split was fixed and the
fold assignment checksummed** before any modelling. Every out-of-fold vector the
repo saved reproduces its recorded fold-mean to within 5e-7, so any two of them can
be blended and compared row for row, months apart. Half of what follows would have
been unmeasurable without that.

## The data is synthetic, and that decided the feature strategy

Playground Series data is generated from an original public dataset. Community
forensic work on this one (credit to Busya PRIME and broccoli beef in the
competition discussion) reported that the original is close to a lookup table:
`addicted` is 1 when `daily_screen_time_hours > 8` or `social_media_hours > 4`.

I did not verify those rules myself, so treat them as reported rather than
established. They score about **0.9888 AUC on the original data and about
0.835 on the synthetic data we are actually scored on.**

That single pair of numbers rewrote my plan. My feature list was a stack of
threshold and ratio features aimed squarely at recovering those rules — and an
untuned LightGBM already scores 0.9549, well above the 0.835 ceiling the rules
themselves reach here. The generator has smeared the decision boundary into
something smooth, and encoding the original rules would have been a step backwards.

**Read the discussion tab before writing features.** It cost me twenty minutes and
saved a week of the most obvious possible dead end.

## My top-priority idea, killed in ninety seconds

Every feature in this dataset has missing values, between 4% and 20%. That is a
conspicuous amount of structure, and missingness indicators were the first thing on
my list — the reasoning being that whether someone declined to report their screen
time might say more than the number itself.

It is a testable claim and it costs one pass over the data, so I tested it before
spending a training run: for each feature, is the target rate different when that
feature is missing?

In [ ]:
mi = sorted(D["missingness"], key=lambda m: m["lift"])
names = [m["feature"] for m in mi]
lift = np.array([m["lift"] for m in mi])
# Each feature's own standard error, recovered from its z. A single shared band
# would have to be drawn at the widest feature's error, which would understate the
# certainty on the other eleven and flatter the conclusion.
se = np.array([abs(m["lift"] / m["z"]) if m["z"] else np.nan for m in mi])

fig, ax = plt.subplots(figsize=(9, 5))
yy = np.arange(len(names))
ax.axvline(0, color=AXIS, lw=1.2, zorder=1)
ax.errorbar(lift, yy, xerr=2 * se, fmt="o", color=BLUE, ms=8, mec=SURFACE,
            mew=1.5, ecolor=MUTED, elinewidth=1.4, capsize=3.5, capthick=1.4,
            zorder=3, label="±2 standard errors")
ax.set_yticks(yy)
ax.set_yticklabels(names, fontsize=9.5, color=SECOND)
ax.set_xlabel("target rate when the feature is missing, minus when it is present")
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND)
ax.grid(axis="y", visible=False)
finish(ax, "Missingness carries no signal about the target",
       "Eleven of twelve intervals cross zero. The largest |z| is 2.24, against the\n"
       "2.4 you would expect from pure noise across twelve tests.")
plt.tight_layout()
plt.show()

print(f"largest |z|: {max(abs(m['z']) for m in mi):.2f}")

Not one feature clears the bar. The largest |z| across twelve tests is **2.24**,
and the largest value you would *expect* from twelve draws of pure noise is about
2.4. `app_opens_per_day` is the one interval that excludes zero, which is what one
in twelve looks like when nothing is going on.

The missingness was injected at random with respect to the target. LightGBM already
routes NaN natively, so twelve indicator columns would have added twelve columns of
nothing.

**This is the part I would most like people to copy.** The idea was wrong, and being
wrong cost ninety seconds instead of a training run and a submission, because it was
phrased as a measurable claim before it was phrased as a feature.

## Adversarial validation: mild, diffuse, not actionable

Train a classifier to tell train rows from test rows. If it succeeds, the two sets
differ and your CV is measuring the wrong distribution.

It scored **AUC 0.562868 +/- 0.000792**. Real but mild, and importance was spread
almost evenly across all eight numeric features at 10-11% each. A concentrated shift
gives you a feature to drop. A diffuse one gives you nothing to act on, so I noted
it and moved on rather than reaching for fold weighting.

Worth running anyway: it is ten minutes, and had it come back at 0.75 the entire
validation scheme would have needed rethinking before anything else was worth doing.

## The model, and the only two things that moved it

LightGBM on the raw twelve features, native categorical handling, native NaN
handling. No feature engineering, because none of it ever helped.

Two changes account for the whole +0.0089:

| change | CV | gain |
|---|---|---|
| untuned defaults, 100 trees | 0.954947 | anchor |
| 1000 trees | 0.962141 | +0.007194 |
| lr 0.05, 2000 trees | 0.963210 | +0.001069 |
| bagging 0.8/0.8, 5 seeds, rank averaged | 0.963880 | +0.000670 |

The first row is the embarrassing one and it is the most useful. The untuned
baseline was **badly underfit**, and 80% of everything I gained in four weeks was
recovered by asking for more trees. Fit the capacity before doing anything clever.

Tuning stopped after the learning rate. The step from lr 0.05 to 0.03 was +0.000064
at 60% more runtime, which is nothing, and the curve had visibly flattened.

In [ ]:
# One seed of the final configuration, retrained here so a ledger row reproduces
# in front of you rather than being asserted. This is the slow cell: five folds of
# 2000 trees over 691k rows, so expect tens of minutes on a Kaggle CPU kernel.
# Set to False to skip and just read the recorded numbers.
RUN_TRAINING = True

RECORDED_SEED42_CV = 0.963471   # experiments.csv row 9

# Do not expect this to reproduce exactly, and do not read a small difference as a bug.
# LightGBM's `deterministic=True` pins the result for a GIVEN THREAD COUNT, not across
# thread counts, because the flag fixes the order in which per-thread gradient sums are
# reduced and that order depends on how many threads there are. Measured on one fold of
# this exact config on the machine the ledger was produced on: n_jobs=6 reproduces the
# recorded value to 3.6e-7, while n_jobs=-1 (8 threads there) lands 3.0e-5 away, and
# both repeat to ~3e-7 across runs. Kaggle's core count is a third thing again.
#
# 1e-4 is the honest bar. It is well below the smallest gain in the ledger (+0.000238)
# and far below the +0.0089 the whole competition was worth, so nothing here rests on
# the fourth decimal of a single run.
TOLERANCE = 1e-4

PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.05, n_estimators=2000,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1,
    # Determinism. Without these two, two runs of the same config differ in the
    # fourth decimal, which is the same size as most of the gains being chased.
    deterministic=True, force_row_wise=True,
)

if RUN_TRAINING:
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold

    from pathlib import Path

    def find_train():
        kag = Path("/kaggle/input/playground-series-s6e8/train.csv")
        if kag.exists():
            return kag
        for b in [Path.cwd(), *Path.cwd().parents]:
            p = b / "data" / "raw" / "train.csv"
            if p.exists():
                return p
        raise FileNotFoundError("train.csv not found")

    train = pd.read_csv(find_train())
    TARGET = "addicted_label"
    CAT = ["gender", "stress_level", "academic_work_impact"]
    feats = [c for c in train.columns if c not in ("id", TARGET)]
    X = train[feats].copy()
    for c in CAT:
        X[c] = X[c].astype("category")
    y = train[TARGET].to_numpy()

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for f, (tr, va) in enumerate(skf.split(X, y)):
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        scores.append(roc_auc_score(y[va], p))
        print(f"  fold {f}: {scores[-1]:.6f}")

    cv = float(np.mean(scores))
    diff = cv - RECORDED_SEED42_CV
    print(f"\nCV {cv:.6f} +/- {np.std(scores):.6f}")
    print(f"recorded in the ledger: {RECORDED_SEED42_CV:.6f}")
    print(f"difference: {diff:+.2e}  (expected: within {TOLERANCE:.0e})")
    print("reproduced" if abs(diff) < TOLERANCE else
          "OUTSIDE TOLERANCE - that would be worth investigating")
else:
    print(f"skipped. recorded CV for this config: {RECORDED_SEED42_CV:.6f}")

## Seed averaging, and where it stops paying

With `subsample=0.8` the model becomes stochastic, so the same config under a
different seed is a genuinely different model. Rank-averaging several of them is the
cheapest ensembling there is: no new ideas, no new features, just the same run again.

One warning first. **A single bagged run does not establish that bagging helps.**
Across five seeds the gain over the identical unbagged config was +0.000261,
+0.000024, +0.000235, +0.000127 and +0.000273. One seed got essentially nothing and
one finished *below* the unbagged model. Had I run one seed and read the result, I
would have believed whichever number I happened to draw. Only the blend is solid.

In [ ]:
sat = D["seed_saturation"]
n = [s["n_seeds"] for s in sat]
v = [s["cv"] for s in sat]

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(n, v, color=BLUE, lw=2, marker="o", ms=8, mec=SURFACE, mew=1.5, zorder=3)
for xi, yi in zip(n, v):
    ax.annotate(f"{yi:.6f}", (xi, yi), textcoords="offset points", xytext=(0, 11),
                ha="center", fontsize=9, color=SECOND)
for i in range(1, len(n)):
    ax.annotate(f"+{(v[i] - v[i-1]) * 1e6:.0f}e-6",
                ((n[i] + n[i-1]) / 2, (v[i] + v[i-1]) / 2),
                textcoords="offset points", xytext=(0, -18), ha="center",
                fontsize=8.5, color=MUTED)
ax.set_xticks(n)
ax.set_xlabel("seeds in the rank-average blend")
ax.set_ylabel("cross-validation ROC AUC")
ax.set_ylim(min(v) - 0.00008, max(v) + 0.00012)
finish(ax, "Seed averaging saturates at four",
       "Each step is roughly half the last.")
plt.tight_layout()
plt.show()

Each step is about half the one before. Seeds six and seven would together buy
roughly 0.00003, so I stopped at five.

One negative result worth recording: **seed averaging did not reduce fold spread**,
0.000555 against 0.000549 for the best single model. The variance-reduction argument
for ensembling failed to show up three separate times in this competition. I have
stopped repeating it.

## The finding I did not expect: correlation does not predict blend value

The standard advice for ensembling is to look for models that are individually
decent and *uncorrelated*, because decorrelated errors cancel. I gated my whole
diversity plan on that, with thresholds written into a notebook before it ran.

So I measured it. Every pair of the twelve models I had saved out-of-fold
predictions for — 66 pairs — scored as a 50/50 rank blend, against the Spearman
correlation between the two members. Gain is measured against **the better of the
two members**, which is the only comparison that answers "was blending worth it".

In [ ]:
pairs = D["pairs"]
BAGGED = {"bagged seed 42", "bagged seed 2024", "bagged seed 7",
          "bagged seed 2025", "bagged seed 13"}
# The clean subset: pairs where the two members share an identical configuration
# and differ only in the random seed. Nothing varies but the stochasticity, so
# nothing is confounded with correlation.
homog = [p for p in pairs if p["a"] in BAGGED and p["b"] in BAGGED]
rest = [p for p in pairs if not (p["a"] in BAGGED and p["b"] in BAGGED)]

r = lambda xs, ys: float(np.corrcoef(xs, ys)[0, 1])
r_all = r([p["spearman"] for p in pairs], [p["gain"] for p in pairs])
r_hom = r([p["spearman"] for p in homog], [p["gain"] for p in homog])
r_gap = r([p["cv_gap"] for p in pairs], [p["gain"] for p in pairs])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

ax = axes[0]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([p["spearman"] for p in rest], [p["gain"] for p in rest], "o",
        color=MUTED, ms=6, alpha=0.55, lw=0, zorder=2, label="all other pairs")
ax.plot([p["spearman"] for p in homog], [p["gain"] for p in homog], "o",
        color=BLUE, ms=8, mec=SURFACE, mew=1.5, lw=0, zorder=3,
        label="identical config, seed only")
ax.set_xlabel("Spearman correlation between the two models")
ax.set_ylabel("blend gain over the better member")
ax.legend(frameon=False, loc="lower left", labelcolor=SECOND, fontsize=9)
finish(ax, "Correlation does not predict blend value",
       f"all 66 pairs r = {r_all:+.2f}   ·   the 10 clean pairs r = {r_hom:+.2f}")

ax = axes[1]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([p["cv_gap"] for p in pairs], [p["gain"] for p in pairs], "o",
        color=BLUE, ms=6.5, mec=SURFACE, mew=1, alpha=0.9, lw=0, zorder=2)
ax.set_xlabel("difference in CV between the two models")
ax.set_ylabel("blend gain over the better member")
finish(ax, "Relative strength does, partly by definition",
       f"r = {r_gap:+.2f}, and a fixed 50/50 weight forces much of it")
plt.tight_layout()
plt.show()

print(f"spearman vs gain, all 66 pairs        : {r_all:+.3f}")
print(f"spearman vs gain, 10 seed-only pairs  : {r_hom:+.3f}")
print(f"cv gap vs gain,   all 66 pairs        : {r_gap:+.3f}")

**Left panel: correlation tells you essentially nothing.** Across all 66 pairs the
relationship between Spearman and blend gain is r = +0.14. The single best blend I
found came from one of the *most* correlated pairs.

**Right panel needs a caveat, and it is important.** Relative strength tracks blend
gain at r = -0.99, but a good part of that is definitional rather than discovered:
at a fixed 50/50 weight, averaging in a much weaker model *has* to drag the result
down. Read the right panel as a constraint the arithmetic imposes, not as a finding.

The honest test of the correlation claim is the blue points — ten pairs of models
with **identical configuration differing only in the random seed**, where nothing is
confounded with correlation. There, Spearman ranges over 0.990 to 0.996, blend gain
ranges over +0.000225 to +0.000360, and the relationship between them is **r =
+0.32 on ten points**, which is noise, and pointing the wrong way for the folk rule.

I want to be careful about what this does and does not license. It is one synthetic
tabular dataset and one model family. It is not a claim that decorrelation is
useless in general. What it does establish, for this data, is that **Spearman was not
a usable gate**, and the practical consequence is concrete:

> CatBoost correlated with my LightGBM at 0.9877 — squarely inside the 0.974 to
> 0.998 band that LightGBM produces against *itself* — and blended **negatively**, at
> -0.000250. The bagged seeds correlate *higher*, at 0.990 to 0.996, and blend
> **positively**, at +0.000546.

My original notebook would have read CatBoost's 0.9877 as "marginal diversity,
proceed" and spent the next day on a five-fold CatBoost run at nine times
LightGBM's cost, followed by XGBoost. Measuring the blend directly cost one fold and
stopped that.

**Gate on the measured blend AUC. It is one number and you can always afford it.**

## Fold standard deviation is the wrong bar for a paired comparison

A rule I had written down for myself, and which is good general advice: do not
believe an improvement smaller than the fold-to-fold spread.

By that rule my one real ensembling result was inconclusive. The seed blend gained
**+0.000546** over the previous best against a fold spread of **0.000549**.

The rule is wrong here, and it is worth understanding why. The per-fold differences
were:

```
+0.000588  +0.000631  +0.000476  +0.000620  +0.000419
```

Five folds out of five, with a standard deviation across those differences of
**0.000094** — a sixth of the fold spread.

Fold-to-fold variation is mostly driven by *which rows landed in which fold*. That
variation is common to both models being compared, so it **cancels in the
difference**. Comparing a paired improvement against the unpaired fold spread throws
away the pairing and inflates your bar by roughly six times.

**The right bar for "is model B better than model A" is the spread of the per-fold
differences, plus how many folds it wins.** By that bar the result is unambiguous.
It nearly got discarded.

## What CV and the public leaderboard actually did

In [ ]:
led = D["ledger"]
x = [r["id"] for r in led]
cv = [r["cv_mean"] for r in led]
#  matters: an unsubmitted row carries a null here, and a bare
# truthiness test would let it through and rely on matplotlib silently dropping it.
lb_x = [r["id"] for r in led if r["lb_public"] is not None]
lb_y = [r["lb_public"] for r in led if r["lb_public"] is not None]

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(x, cv, color=BLUE, lw=2, marker="o", ms=5.5, label="cross-validation",
        zorder=3, mec=SURFACE, mew=1.5)
ax.plot(lb_x, lb_y, color=ORANGE, lw=0, marker="D", ms=7, mec=SURFACE, mew=1.5,
        label="public leaderboard", zorder=4)
for xi, yi in zip(lb_x, lb_y):
    ax.annotate(f"{yi:.5f}", (xi, yi), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=8.5, color=SECOND)
ax.annotate("untuned anchor", (1, cv[0]), textcoords="offset points",
            xytext=(10, 7), fontsize=9, color=SECOND)
ax.set_xlabel("experiment")
ax.set_ylabel("ROC AUC")
ax.set_xticks(x)
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND)
finish(ax, "Every experiment in the ledger",
       "The whole competition is +0.0089 AUC, and all of it is model capacity.")
plt.tight_layout()
plt.show()

The leaderboard sits consistently above CV by roughly 0.0012 and moves in the same
direction, which is all you need from a validation scheme.

The interesting part is the last two submissions. My 3-seed and 5-seed blends are
separated **cleanly** by CV: the 5-seed wins 5 folds out of 5 with a paired standard
deviation of 0.000023. On the public leaderboard they scored **0.965090** and
**0.965080** — a gap of 0.00001, in the other direction.

That is not a CV/LB disagreement to diagnose. The public leaderboard here is a
sample, and its standard error is near **0.001** — larger than every single gain
after the fourth experiment. It is a 6e-5 effect measured with a 1e-3 ruler.

The practical consequence for final submission selection: the usual split is
best-CV plus best-public-LB, but when the public LB's noise floor exceeds the
effects you are choosing between, **best-public-LB is close to picking at random**.
CV gets the heavier weight here.

## The ledger, and what I would do differently

Fifteen experiments, every one written down including the failures. The failures
are most of the value: they are why the same dead end did not get walked twice.

**What worked**

- Fitting model capacity first. 80% of the total gain.
- Lower learning rate with a compensating tree count. A small gain, and a useful
  side effect: fold spread fell from 0.000816 to 0.000549, so every later comparison
  got easier to call.
- Bagged seed averaging, rank-blended, up to four seeds.

**What did not**

- Missingness indicators. No signal, established in ninety seconds.
- Threshold and rule features. The generator caps them below the untuned baseline.
- External data. The original is 7,500 rows against 691,369, its distributions are
  warped relative to the synthetic, and it appears to be synthetic itself.
- CatBoost. 0.0017 behind at matched budget, blends negatively, nine times the cost.
- Capacity diversity within LightGBM. Blends negatively.

**What I would do differently**

1. **Time one fold before launching anything.** I started a CatBoost run estimated
   at 30-50 minutes. It ran for two hours with `verbose=0` and no progress output
   before being killed. It was never hung — a single fold was ten minutes and the
   notebook called the pipeline twice. The estimate was wrong by 3x because no
   fold had been timed.
2. **Not offer a mechanism from one data point.** I attributed the CV/LB gap to fold
   averaging on the first submission, patched the story on the second, and withdrew
   it on the third when the pattern did not hold.
3. **Check the row count before calling something a lever.** I described the external
   dataset as one of the few real levers available, then found it was a 1% increase
   in training data.

**What I would keep**

The checksummed fold split. It is ten lines and it is the reason a vector saved in
week one could be blended against one from week four with any confidence at all.

---

Ledger, notebooks and working log: **github.com/vyask21/smartphone-addiction**

If one thing here is worth taking away, it is that a cheap test of a wrong idea is
worth more than an expensive test of a good one, and most ideas are wrong.